In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

import pandas as pd

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/59281338-7aa6-4b99-8f53-e59317ae67a7/nltk_data..
[nltk_data]     .
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df = pd.read_csv('npr.csv')
documents = df['Article'].tolist()

print(df.head())
print()
print("Total documents:", len(documents))

                                             Article
0  In the Washington of 2016, even when the polic...
1    Donald Trump has used Twitter  —   his prefe...
2    Donald Trump is unabashedly praising Russian...
3  Updated at 2:50 p. m. ET, Russian President Vl...
4  From photography, illustration and video, to d...

Total documents: 11992


In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())                 # convert to lowercase and tokenize
    tokens = [token for token in tokens if token.isalnum()]   # remove non-alphanumeric tokens
    tokens = [token for token in tokens if token not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]       # lemmatize tokens
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

print(preprocessed_documents[0])

['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker', 'eye', 'gop', 'leader', 'sounded', 'much', 'theme', 'urging', 'president', 'obama', 'year', 'take', 'strong', 'action', 'deter', 'russia', 'worldwide', 'aggression', 'including', 'operation', 'wrote', 'devin', 'nunes', 'chairman', 'house', 'intelligence', 'committee', 'week', 'left', 'office', 'president', 'suddenly', 'decided', 'stronger', 'measure', 'indeed', 'appearing', 'cnn', 'frequent', 'obama', 'critic', 'trent', 'frank', 'called', 'much', 'tougher', 'acti

In [4]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

# Convert each preprocessed document into bag-of-words representation
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print("Number of unique tokens:", len(dictionary))
print("First corpus item:", corpus[0])

Number of unique tokens: 15966
First corpus item: [(0, 1), (1, 3), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 4), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 3), (16, 1), (17, 1), (18, 2), (19, 1), (20, 2), (21, 2), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 2), (36, 2), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 1), (49, 2), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 6), (56, 4), (57, 1), (58, 1), (59, 2), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 2), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 1), (72, 5), (73, 1), (74, 1), (75, 1), (76, 1), (77, 3), (78, 2), (79, 1), (80, 1), (81, 1), (82, 1), (83, 1), (84, 1), (85, 1), (86, 1), (87, 1), (88, 1), (89, 1), (90, 1), (91, 1), (92, 1), (93, 1), (94, 1), (95, 1), (96, 1), (97, 1), (98, 1), (99, 1), (100, 1), (101, 1), (102, 1), (103, 1), (104, 1), (105, 1),

In [5]:
lda_model = LdaModel(
    corpus=corpus,
    num_topics=5,
    id2word=dictionary,
    passes=15,
    random_state=42
)

In [6]:
article_labels = []

for i, doc in enumerate(preprocessed_documents):
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({
    "Article": documents,
    "Topic": article_labels
})

print("Table with Articles and Topic:")
print(df_result)
print()

Table with Articles and Topic:
                                                 Article  Topic
0      In the Washington of 2016, even when the polic...      1
1        Donald Trump has used Twitter  —   his prefe...      0
2        Donald Trump is unabashedly praising Russian...      1
3      Updated at 2:50 p. m. ET, Russian President Vl...      0
4      From photography, illustration and video, to d...      4
...                                                  ...    ...
11987  The number of law enforcement officers shot an...      0
11988    Trump is busy these days with victory tours,...      1
11989  It’s always interesting for the Goats and Soda...      3
11990  The election of Donald Trump was a surprise to...      1
11991  Voters in the English city of Sunderland did s...      1

[11992 rows x 2 columns]



In [7]:
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

Top terms for Topic #0:
['police', 'report', 'state', 'government', 'country', 'law', 'court', 'official', 'told', 'case']

Top terms for Topic #1:
['trump', 'clinton', 'president', 'state', 'republican', 'campaign', 'election', 'obama', 'vote', 'voter']

Top terms for Topic #2:
['know', 'think', 'thing', 'life', 'woman', 'really', 'story', 'show', 'book', 'u']

Top terms for Topic #3:
['health', 'school', 'percent', 'study', 'child', 'student', 'care', 'state', 'program', 'company']

Top terms for Topic #4:
['food', 'water', 'world', 'city', 'day', 'animal', 'around', 'back', 'home', 'company']



In [8]:
print("Top Terms for Each Topic:")

for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "police" (weight: 0.006)
- "report" (weight: 0.006)
- "state" (weight: 0.005)
- "government" (weight: 0.005)
- "country" (weight: 0.005)
- "law" (weight: 0.004)
- "court" (weight: 0.004)
- "official" (weight: 0.004)
- "told" (weight: 0.004)
- "case" (weight: 0.004)

Topic 1:
- "trump" (weight: 0.031)
- "clinton" (weight: 0.012)
- "president" (weight: 0.010)
- "state" (weight: 0.009)
- "republican" (weight: 0.009)
- "campaign" (weight: 0.008)
- "election" (weight: 0.006)
- "obama" (weight: 0.006)
- "vote" (weight: 0.006)
- "voter" (weight: 0.005)

Topic 2:
- "know" (weight: 0.005)
- "think" (weight: 0.005)
- "thing" (weight: 0.005)
- "life" (weight: 0.005)
- "woman" (weight: 0.005)
- "really" (weight: 0.004)
- "story" (weight: 0.004)
- "show" (weight: 0.004)
- "book" (weight: 0.003)
- "u" (weight: 0.003)

Topic 3:
- "health" (weight: 0.009)
- "school" (weight: 0.007)
- "percent" (weight: 0.006)
- "study" (weight: 0.006)
- "child" (weight: 0.005)
- "s

In [9]:
print("Example interpretation:")
print("Topic 0: Education")
print("Topic 1: Personal")
print("Topic 2: World Politics")
print("Topic 3: Health")
print("Topic 4: US Election")

Example interpretation:
Topic 0: Education
Topic 1: Personal
Topic 2: World Politics
Topic 3: Health
Topic 4: US Election
